<a href="https://colab.research.google.com/github/MadhawaRathnayake/DataPreprocessing-HPC/blob/cuda_analyzer/modules/analyzer_cuda/cuda_analyzer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/MadhawaRathnayake/DataPreprocessing-HPC.git
%cd DataPreprocessing-HPC
!git branch -a


Cloning into 'DataPreprocessing-HPC'...
remote: Enumerating objects: 183, done.
remote: Counting objects: 100% (183/183), done.
remote: Compressing objects: 100% (86/86), done.
remote: Total 183 (delta 97), reused 181 (delta 95), pack-reused 0 (from 0)
Receiving objects: 100% (183/183), 2.60 MiB | 14.61 MiB/s, done.
Resolving deltas: 100% (97/97), done.
/content/DataPreprocessing-HPC
* main
  remotes/origin/HEAD -> origin/main
  remotes/origin/Pipeline-Wired-up
  remotes/origin/cuda_analyzer
  remotes/origin/main
  remotes/origin/series-openmp-mpi-integration


In [ ]:
!git checkout -b cuda_analyzer origin/cuda_analyzer


Branch 'cuda_analyzer' set up to track remote branch 'cuda_analyzer' from 'origin'.
Switched to a new branch 'cuda_analyzer'


In [ ]:
!git branch
!git status

* cuda_analyzer
  main
On branch cuda_analyzer
Your branch is up to date with 'origin/cuda_analyzer'.

nothing to commit, working tree clean


In [ ]:
!git branch
!pwd
!ls

* cuda_analyzer
  main
/content/DataPreprocessing-HPC
build.sh  MODULAR_UI_CHANGES.md  QUICKSTART.md	SKILL.md
data	  modules		 README.md	ui
lib	  PROJECT_OVERVIEW.md	 run.sh


In [ ]:
%cd /content/DataPreprocessing-HPC/modules
!ls

/content/DataPreprocessing-HPC/modules
analyzer_cuda  analyzer_mpi  analyzer_openmp  importer	series_processing


In [ ]:
%cd /content/DataPreprocessing-HPC/modules/analyzer_cuda
!ls

/content/DataPreprocessing-HPC/modules/analyzer_cuda
cuda_analyzer.ipynb


In [ ]:
import os
os.chdir('/content/DataPreprocessing-HPC/modules/analyzer_cuda')
print(os.getcwd())

/content/DataPreprocessing-HPC/modules/analyzer_cuda


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content/DataPreprocessing-HPC/modules/analyzer_cuda

/content/DataPreprocessing-HPC/modules/analyzer_cuda


In [ ]:
!find /content/DataPreprocessing-HPC -name "cuda_analyzer.ipynb"

/content/DataPreprocessing-HPC/modules/analyzer_cuda/cuda_analyzer.ipynb


In [ ]:
import os
os.chdir('/content/DataPreprocessing-HPC/modules/analyzer_cuda')
print(os.getcwd())
!ls -la

/content/DataPreprocessing-HPC/modules/analyzer_cuda
total 8
drwxr-xr-x 2 root root 4096 Mar  9 00:22 .
drwxr-xr-x 7 root root 4096 Mar  9 00:22 ..
-rw-r--r-- 1 root root    0 Mar  9 00:22 cuda_analyzer.ipynb


In [ ]:
import json

nb = {
    "cells": [
        {
            "cell_type": "code",
            "execution_count": None,
            "metadata": {},
            "outputs": [],
            "source": [
                "# Start coding here\n",
                "print('Hello from cuda_analyzer')\n"
            ]
        }
    ],
    "metadata": {
        "colab": {"name": "cuda_analyzer.ipynb"},
        "kernelspec": {
            "display_name": "Python 3",
            "name": "python3"
        },
        "language_info": {"name": "python"}
    },
    "nbformat": 4,
    "nbformat_minor": 0
}

path = "/content/DataPreprocessing-HPC/modules/analyzer_cuda/cuda_analyzer.ipynb"
with open(path, "w") as f:
    json.dump(nb, f)

print("Created valid notebook at:", path)

Created valid notebook at: /content/DataPreprocessing-HPC/modules/analyzer_cuda/cuda_analyzer.ipynb


In [2]:
%%writefile /cuda_analyzer.h
#ifndef CUDA_ANALYZER_H
#define CUDA_ANALYZER_H

#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <strings.h>
#include <math.h>
#include <ctype.h>
#include <cuda_runtime.h>

#define MAX_UNIQUE_VALUES 1000
#define MAX_OUTLIERS 100

typedef enum {
    TYPE_NUMERIC,
    TYPE_CATEGORICAL,
    TYPE_MIXED,
    TYPE_UNKNOWN
} DataType;

typedef enum {
    CAT_CONTINUOUS,
    CAT_DISCRETE,
    CAT_NOMINAL,
    CAT_ORDINAL,
    CAT_BINARY,
    CAT_UNKNOWN
} Category;

typedef struct {
    char *value;
    int count;
} ValueCount;

typedef struct {
    char *column_name;
    DataType data_type;
    Category category;

    int total_count;
    int null_count;
    int unique_count;
    double null_percentage;

    double min_value;
    double max_value;
    double mean;
    double median;
    double std_dev;
    double *outliers;
    int outlier_count;

    ValueCount *value_counts;
    int value_count_size;

    int has_nulls;
    int has_outliers;
    int has_duplicates;
    int type_consistent;
} ColumnStats;

typedef struct {
    ColumnStats *columns;
    int num_columns;
    double processing_time;
    int gpu_used;
} DatasetStats;

DatasetStats* analyzer_cuda_create_stats(int num_columns);
void analyzer_cuda_free_stats(DatasetStats *stats);
int analyzer_cuda_analyze_dataset(char **data, char **headers, int num_rows,
                                  int num_cols, DatasetStats *stats);
void analyzer_cuda_print_stats(DatasetStats *stats);
char* analyzer_cuda_get_stats_json(DatasetStats *stats);

#endif

Writing /cuda_analyzer.h


In [3]:
%%writefile /cuda_analyzer.h
#ifndef CUDA_ANALYZER_H
#define CUDA_ANALYZER_H

#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <strings.h>
#include <math.h>
#include <ctype.h>
#include <cuda_runtime.h>

#define MAX_UNIQUE_VALUES 1000
#define MAX_OUTLIERS 100

typedef enum {
    TYPE_NUMERIC,
    TYPE_CATEGORICAL,
    TYPE_MIXED,
    TYPE_UNKNOWN
} DataType;

typedef enum {
    CAT_CONTINUOUS,
    CAT_DISCRETE,
    CAT_NOMINAL,
    CAT_ORDINAL,
    CAT_BINARY,
    CAT_UNKNOWN
} Category;

typedef struct {
    char *value;
    int count;
} ValueCount;

typedef struct {
    char *column_name;
    DataType data_type;
    Category category;

    int total_count;
    int null_count;
    int unique_count;
    double null_percentage;

    double min_value;
    double max_value;
    double mean;
    double median;
    double std_dev;
    double *outliers;
    int outlier_count;

    ValueCount *value_counts;
    int value_count_size;

    int has_nulls;
    int has_outliers;
    int has_duplicates;
    int type_consistent;
} ColumnStats;

typedef struct {
    ColumnStats *columns;
    int num_columns;
    double processing_time;
    int gpu_used;
} DatasetStats;

DatasetStats* analyzer_cuda_create_stats(int num_columns);
void analyzer_cuda_free_stats(DatasetStats *stats);
int analyzer_cuda_analyze_dataset(char **data, char **headers, int num_rows,
                                  int num_cols, DatasetStats *stats);
void analyzer_cuda_print_stats(DatasetStats *stats);
char* analyzer_cuda_get_stats_json(DatasetStats *stats);

#endif

Overwriting /cuda_analyzer.h


In [4]:
%%writefile /cuda_analyzer.cu
#include "cuda_analyzer.h"
#include <float.h>
#include <thrust/device_vector.h>
#include <thrust/sort.h>

static int is_numeric(const char *str) {
    if (!str || *str == '\0') return 0;
    char *endptr;
    strtod(str, &endptr);
    while (*endptr && isspace((unsigned char)*endptr)) endptr++;
    return *endptr == '\0';
}

static int is_null(const char *str) {
    if (!str) return 1;
    while (*str && isspace((unsigned char)*str)) str++;
    if (*str == '\0') return 1;

    if (strcasecmp(str, "null") == 0 ||
        strcasecmp(str, "na") == 0 ||
        strcasecmp(str, "n/a") == 0 ||
        strcasecmp(str, "nan") == 0) {
        return 1;
    }
    return 0;
}

static int compare_value_counts(const void *a, const void *b) {
    return ((ValueCount*)b)->count - ((ValueCount*)a)->count;
}

__device__ double atomicMinDouble(double* address, double val) {
    unsigned long long int* address_as_ull = (unsigned long long int*)address;
    unsigned long long int old = *address_as_ull, assumed;

    do {
        assumed = old;
        old = atomicCAS(address_as_ull, assumed,
                        __double_as_longlong(fmin(val, __longlong_as_double(assumed))));
    } while (assumed != old);

    return __longlong_as_double(old);
}

__device__ double atomicMaxDouble(double* address, double val) {
    unsigned long long int* address_as_ull = (unsigned long long int*)address;
    unsigned long long int old = *address_as_ull, assumed;

    do {
        assumed = old;
        old = atomicCAS(address_as_ull, assumed,
                        __double_as_longlong(fmax(val, __longlong_as_double(assumed))));
    } while (assumed != old);

    return __longlong_as_double(old);
}

__global__ void reduce_stats_kernel(const double *data, int n,
                                    double *d_min, double *d_max,
                                    double *d_sum, double *d_sumsq) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) {
        double v = data[i];
        atomicMinDouble(d_min, v);
        atomicMaxDouble(d_max, v);
        atomicAdd(d_sum, v);
        atomicAdd(d_sumsq, v * v);
    }
}

DatasetStats* analyzer_cuda_create_stats(int num_columns) {
    if (num_columns <= 0) return NULL;

    DatasetStats *stats = (DatasetStats*)malloc(sizeof(DatasetStats));
    if (!stats) return NULL;

    stats->num_columns = num_columns;
    stats->columns = (ColumnStats*)calloc(num_columns, sizeof(ColumnStats));
    stats->processing_time = 0.0;
    stats->gpu_used = 0;

    if (!stats->columns) {
        free(stats);
        return NULL;
    }

    return stats;
}

void analyzer_cuda_free_stats(DatasetStats *stats) {
    if (!stats) return;

    for (int i = 0; i < stats->num_columns; i++) {
        free(stats->columns[i].column_name);
        free(stats->columns[i].outliers);

        if (stats->columns[i].value_counts) {
            for (int j = 0; j < stats->columns[i].value_count_size; j++) {
                free(stats->columns[i].value_counts[j].value);
            }
            free(stats->columns[i].value_counts);
        }
    }

    free(stats->columns);
    free(stats);
}

static int analyze_numeric_cuda(double *numeric_values, int numeric_count, ColumnStats *stats) {
    if (numeric_count <= 0) return -1;

    double *d_data = NULL, *d_min = NULL, *d_max = NULL, *d_sum = NULL, *d_sumsq = NULL;

    cudaMalloc((void**)&d_data, numeric_count * sizeof(double));
    cudaMalloc((void**)&d_min, sizeof(double));
    cudaMalloc((void**)&d_max, sizeof(double));
    cudaMalloc((void**)&d_sum, sizeof(double));
    cudaMalloc((void**)&d_sumsq, sizeof(double));

    double init_min = DBL_MAX;
    double init_max = -DBL_MAX;
    double init_zero = 0.0;

    cudaMemcpy(d_data, numeric_values, numeric_count * sizeof(double), cudaMemcpyHostToDevice);
    cudaMemcpy(d_min, &init_min, sizeof(double), cudaMemcpyHostToDevice);
    cudaMemcpy(d_max, &init_max, sizeof(double), cudaMemcpyHostToDevice);
    cudaMemcpy(d_sum, &init_zero, sizeof(double), cudaMemcpyHostToDevice);
    cudaMemcpy(d_sumsq, &init_zero, sizeof(double), cudaMemcpyHostToDevice);

    int threads = 256;
    int blocks = (numeric_count + threads - 1) / threads;
    reduce_stats_kernel<<<blocks, threads>>>(d_data, numeric_count, d_min, d_max, d_sum, d_sumsq);
    cudaDeviceSynchronize();

    double min_val, max_val, sum, sumsq;
    cudaMemcpy(&min_val, d_min, sizeof(double), cudaMemcpyDeviceToHost);
    cudaMemcpy(&max_val, d_max, sizeof(double), cudaMemcpyDeviceToHost);
    cudaMemcpy(&sum, d_sum, sizeof(double), cudaMemcpyDeviceToHost);
    cudaMemcpy(&sumsq, d_sumsq, sizeof(double), cudaMemcpyDeviceToHost);

    stats->min_value = min_val;
    stats->max_value = max_val;
    stats->mean = sum / numeric_count;

    double variance = (sumsq / numeric_count) - (stats->mean * stats->mean);
    if (variance < 0.0) variance = 0.0;
    stats->std_dev = sqrt(variance);

    thrust::device_vector<double> dvec(numeric_values, numeric_values + numeric_count);
    thrust::sort(dvec.begin(), dvec.end());
    thrust::copy(dvec.begin(), dvec.end(), numeric_values);

    if (numeric_count % 2 == 0) {
        stats->median = (numeric_values[numeric_count / 2 - 1] +
                         numeric_values[numeric_count / 2]) / 2.0;
    } else {
        stats->median = numeric_values[numeric_count / 2];
    }

    int q1_idx = numeric_count / 4;
    int q3_idx = (3 * numeric_count) / 4;
    double q1 = numeric_values[q1_idx];
    double q3 = numeric_values[q3_idx];
    double iqr = q3 - q1;
    double lower_bound = q1 - 1.5 * iqr;
    double upper_bound = q3 + 1.5 * iqr;

    stats->outliers = (double*)malloc(MAX_OUTLIERS * sizeof(double));
    stats->outlier_count = 0;
    stats->has_outliers = 0;

    if (stats->outliers) {
        for (int i = 0; i < numeric_count && stats->outlier_count < MAX_OUTLIERS; i++) {
            if (numeric_values[i] < lower_bound || numeric_values[i] > upper_bound) {
                stats->outliers[stats->outlier_count++] = numeric_values[i];
            }
        }
        stats->has_outliers = (stats->outlier_count > 0);
    }

    cudaFree(d_data);
    cudaFree(d_min);
    cudaFree(d_max);
    cudaFree(d_sum);
    cudaFree(d_sumsq);

    return 0;
}

static void analyze_column(char **column_data, int num_rows,
                           const char *col_name, ColumnStats *stats) {
    stats->column_name = (char*)malloc(strlen(col_name) + 1);
    strcpy(stats->column_name, col_name);

    stats->total_count = num_rows;
    stats->null_count = 0;
    stats->unique_count = 0;
    stats->outlier_count = 0;
    stats->has_nulls = 0;
    stats->has_outliers = 0;
    stats->has_duplicates = 0;
    stats->type_consistent = 1;
    stats->outliers = NULL;
    stats->value_counts = NULL;
    stats->value_count_size = 0;

    double *numeric_values = (double*)malloc(num_rows * sizeof(double));
    char **unique_values = (char**)malloc(num_rows * sizeof(char*));
    int *unique_counts = (int*)calloc(num_rows, sizeof(int));

    int numeric_count = 0;
    int categorical_count = 0;

    for (int i = 0; i < num_rows; i++) {
        char *value = column_data[i];

        if (is_null(value)) {
            stats->null_count++;
            continue;
        }

        if (is_numeric(value)) {
            numeric_values[numeric_count++] = atof(value);
        } else {
            categorical_count++;
        }

        int found = 0;
        for (int j = 0; j < stats->unique_count; j++) {
            if (strcmp(unique_values[j], value) == 0) {
                unique_counts[j]++;
                found = 1;
                break;
            }
        }

        if (!found) {
            unique_values[stats->unique_count] = (char*)malloc(strlen(value) + 1);
            strcpy(unique_values[stats->unique_count], value);
            unique_counts[stats->unique_count] = 1;
            stats->unique_count++;
        }
    }

    stats->null_percentage = (num_rows > 0) ? ((double)stats->null_count / num_rows) * 100.0 : 0.0;
    stats->has_nulls = (stats->null_count > 0);
    stats->has_duplicates = (stats->unique_count < (num_rows - stats->null_count));

    if (numeric_count > 0 && categorical_count == 0) {
        stats->data_type = TYPE_NUMERIC;
    } else if (categorical_count > 0 && numeric_count == 0) {
        stats->data_type = TYPE_CATEGORICAL;
    } else if (numeric_count > 0 && categorical_count > 0) {
        stats->data_type = TYPE_MIXED;
        stats->type_consistent = 0;
    } else {
        stats->data_type = TYPE_UNKNOWN;
    }

    if (stats->data_type == TYPE_NUMERIC && numeric_count > 0) {
        analyze_numeric_cuda(numeric_values, numeric_count, stats);

        if (stats->unique_count == 2) stats->category = CAT_BINARY;
        else if (stats->unique_count < 10) stats->category = CAT_DISCRETE;
        else stats->category = CAT_CONTINUOUS;
    } else {
        stats->min_value = 0.0;
        stats->max_value = 0.0;
        stats->mean = 0.0;
        stats->median = 0.0;
        stats->std_dev = 0.0;
        stats->outlier_count = 0;
        stats->has_outliers = 0;

        if (stats->unique_count == 2) stats->category = CAT_BINARY;
        else if (stats->data_type == TYPE_UNKNOWN) stats->category = CAT_UNKNOWN;
        else stats->category = CAT_NOMINAL;
    }

    stats->value_count_size = (stats->unique_count < 10) ? stats->unique_count : 10;

    if (stats->value_count_size > 0) {
        stats->value_counts = (ValueCount*)malloc(stats->value_count_size * sizeof(ValueCount));
        ValueCount *temp_counts = (ValueCount*)malloc(stats->unique_count * sizeof(ValueCount));

        for (int i = 0; i < stats->unique_count; i++) {
            temp_counts[i].value = unique_values[i];
            temp_counts[i].count = unique_counts[i];
        }

        qsort(temp_counts, stats->unique_count, sizeof(ValueCount), compare_value_counts);

        for (int i = 0; i < stats->value_count_size; i++) {
            stats->value_counts[i].value = (char*)malloc(strlen(temp_counts[i].value) + 1);
            strcpy(stats->value_counts[i].value, temp_counts[i].value);
            stats->value_counts[i].count = temp_counts[i].count;
        }

        free(temp_counts);
    }

    for (int i = 0; i < stats->unique_count; i++) free(unique_values[i]);
    free(unique_values);
    free(unique_counts);
    free(numeric_values);
}

int analyzer_cuda_analyze_dataset(char **data, char **headers, int num_rows,
                                  int num_cols, DatasetStats *stats) {
    if (!data || !headers || !stats) return -1;

    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);
    cudaEventRecord(start);

    int device_count = 0;
    cudaGetDeviceCount(&device_count);
    stats->gpu_used = (device_count > 0) ? 1 : 0;

    printf("Starting CUDA analysis on %d rows x %d columns...\n", num_rows, num_cols);

    for (int col = 0; col < num_cols; col++) {
        char **column_data = (char**)malloc(num_rows * sizeof(char*));
        for (int row = 0; row < num_rows; row++) {
            column_data[row] = data[row * num_cols + col];
        }

        analyze_column(column_data, num_rows, headers[col], &stats->columns[col]);
        free(column_data);

        printf("Analyzed column %d/%d: %s\n", col + 1, num_cols, headers[col]);
    }

    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float ms = 0.0f;
    cudaEventElapsedTime(&ms, start, stop);
    stats->processing_time = ms / 1000.0;

    cudaEventDestroy(start);
    cudaEventDestroy(stop);

    printf("CUDA analysis completed in %.4f seconds. GPU used: %s\n",
           stats->processing_time, stats->gpu_used ? "Yes" : "No");

    return 0;
}

void analyzer_cuda_print_stats(DatasetStats *stats) {
    if (!stats) return;

    printf("\n=== Dataset Statistics (CUDA) ===\n");
    printf("Total columns: %d\n", stats->num_columns);
    printf("Processing time: %.4f seconds\n", stats->processing_time);
    printf("GPU used: %d\n\n", stats->gpu_used);

    for (int i = 0; i < stats->num_columns; i++) {
        ColumnStats *col = &stats->columns[i];

        printf("Column: %s\n", col->column_name);
        printf("  Data Type: ");
        switch (col->data_type) {
            case TYPE_NUMERIC: printf("Numeric\n"); break;
            case TYPE_CATEGORICAL: printf("Categorical\n"); break;
            case TYPE_MIXED: printf("Mixed\n"); break;
            default: printf("Unknown\n");
        }

        printf("  Total Count: %d\n", col->total_count);
        printf("  Null Count: %d (%.2f%%)\n", col->null_count, col->null_percentage);
        printf("  Unique Count: %d\n", col->unique_count);

        if (col->data_type == TYPE_NUMERIC) {
            printf("  Min: %.2f\n", col->min_value);
            printf("  Max: %.2f\n", col->max_value);
            printf("  Mean: %.2f\n", col->mean);
            printf("  Median: %.2f\n", col->median);
            printf("  Std Dev: %.2f\n", col->std_dev);
            printf("  Outliers: %d\n", col->outlier_count);
        }
        printf("\n");
    }
}

char* analyzer_cuda_get_stats_json(DatasetStats *stats) {
    if (!stats) return NULL;

    char *json = (char*)malloc(1000000);
    if (!json) return NULL;

    char *ptr = json;
    ptr += sprintf(ptr, "{\n");
    ptr += sprintf(ptr, "  \"processing_time\": %.4f,\n", stats->processing_time);
    ptr += sprintf(ptr, "  \"gpu_used\": %d,\n", stats->gpu_used);
    ptr += sprintf(ptr, "  \"columns\": [\n");

    for (int i = 0; i < stats->num_columns; i++) {
        ColumnStats *col = &stats->columns[i];

        ptr += sprintf(ptr, "    {\n");
        ptr += sprintf(ptr, "      \"column_name\": \"%s\",\n", col->column_name);
        ptr += sprintf(ptr, "      \"data_type\": \"%s\",\n",
                       col->data_type == TYPE_NUMERIC ? "Numeric" :
                       col->data_type == TYPE_CATEGORICAL ? "Categorical" :
                       col->data_type == TYPE_MIXED ? "Mixed" : "Unknown");
        ptr += sprintf(ptr, "      \"total_count\": %d,\n", col->total_count);
        ptr += sprintf(ptr, "      \"null_count\": %d,\n", col->null_count);
        ptr += sprintf(ptr, "      \"null_percentage\": %.2f,\n", col->null_percentage);
        ptr += sprintf(ptr, "      \"unique_count\": %d,\n", col->unique_count);

        if (col->data_type == TYPE_NUMERIC) {
            ptr += sprintf(ptr, "      \"min_value\": %.2f,\n", col->min_value);
            ptr += sprintf(ptr, "      \"max_value\": %.2f,\n", col->max_value);
            ptr += sprintf(ptr, "      \"mean\": %.2f,\n", col->mean);
            ptr += sprintf(ptr, "      \"median\": %.2f,\n", col->median);
            ptr += sprintf(ptr, "      \"std_dev\": %.2f,\n", col->std_dev);
            ptr += sprintf(ptr, "      \"outlier_count\": %d,\n", col->outlier_count);
        }

        ptr += sprintf(ptr, "      \"has_nulls\": %s,\n", col->has_nulls ? "true" : "false");
        ptr += sprintf(ptr, "      \"has_outliers\": %s,\n", col->has_outliers ? "true" : "false");
        ptr += sprintf(ptr, "      \"has_duplicates\": %s,\n", col->has_duplicates ? "true" : "false");
        ptr += sprintf(ptr, "      \"type_consistent\": %s\n", col->type_consistent ? "true" : "false");
        ptr += sprintf(ptr, "    }%s\n", (i < stats->num_columns - 1) ? "," : "");
    }

    ptr += sprintf(ptr, "  ]\n}\n");
    return json;
}

Writing /cuda_analyzer.cu


In [5]:
%%writefile /test_main.cu
#include "cuda_analyzer.h"

int main() {
    int num_rows = 5;
    int num_cols = 3;

    char *headers[] = {
        (char*)"age",
        (char*)"salary",
        (char*)"city"
    };

    char *data[] = {
        (char*)"20", (char*)"50000",  (char*)"Colombo",
        (char*)"25", (char*)"60000",  (char*)"Kandy",
        (char*)"30", (char*)"55000",  (char*)"Colombo",
        (char*)"35", (char*)"200000", (char*)"Galle",
        (char*)"na", (char*)"62000",  (char*)"Jaffna"
    };

    DatasetStats *stats = analyzer_cuda_create_stats(num_cols);
    if (!stats) {
        printf("Failed to create stats\n");
        return 1;
    }

    analyzer_cuda_analyze_dataset(data, headers, num_rows, num_cols, stats);
    analyzer_cuda_print_stats(stats);

    char *json = analyzer_cuda_get_stats_json(stats);
    if (json) {
        printf("\nJSON Output:\n%s\n", json);
        free(json);
    }

    analyzer_cuda_free_stats(stats);
    return 0;
}

Writing /test_main.cu


In [6]:
!nvidia-smi
!nvcc --version
!nvcc -o /cuda_test /test_main.cu /cuda_analyzer.cu -std=c++11
!/cuda_test

Mon Mar  9 01:03:48 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [7]:
!nvcc -arch=sm_75 -o /cuda_test /test_main.cu /cuda_analyzer.cu -std=c++11
!/cuda_test

Starting CUDA analysis on 5 rows x 3 columns...
Analyzed column 1/3: age
Analyzed column 2/3: salary
Analyzed column 3/3: city
CUDA analysis completed in 0.0029 seconds. GPU used: Yes

=== Dataset Statistics (CUDA) ===
Total columns: 3
Processing time: 0.0029 seconds
GPU used: 1

Column: age
  Data Type: Numeric
  Total Count: 5
  Null Count: 1 (20.00%)
  Unique Count: 4
  Min: 20.00
  Max: 35.00
  Mean: 27.50
  Median: 27.50
  Std Dev: 5.59
  Outliers: 0

Column: salary
  Data Type: Numeric
  Total Count: 5
  Null Count: 0 (0.00%)
  Unique Count: 5
  Min: 50000.00
  Max: 200000.00
  Mean: 85400.00
  Median: 60000.00
  Std Dev: 57451.20
  Outliers: 1

Column: city
  Data Type: Categorical
  Total Count: 5
  Null Count: 0 (0.00%)
  Unique Count: 4


JSON Output:
{
  "processing_time": 0.0029,
  "gpu_used": 1,
  "columns": [
    {
      "column_name": "age",
      "data_type": "Numeric",
      "total_count": 5,
      "null_count": 1,
      "null_percentage": 20.00,
      "unique_count": 

In [8]:
!cp /cuda_analyzer.h /content/
!cp /cuda_analyzer.cu /content/
!cp /test_main.cu /content/